# Stage 1 - score every text with the StressTwin-F ensemble (GPU runtime)

Uses the weights exported by the retrain notebook. Run it twice for the ablation: `SCORER = 'stresstwin_f'`, then `'plain_ft'`.

In [ ]:
# Kaggle setup. Settings panel: Internet ON for stage 0 only; GPU only for stage 1.
# Add your private dataset "stresstwin-p" via Add Input. The folder that contains pyproject.toml is found automatically.
import os, shutil, subprocess, sys
def find_pkg():
    """Find the package wherever Kaggle mounted the dataset (the mount path differs between Kaggle versions), or unzip it if it was left zipped."""
    import zipfile
    for r, dirs, files in os.walk('/kaggle/input'):
        if 'pyproject.toml' in files and os.path.isdir(os.path.join(r, 'stresstwin_p')):
            return r
    for r, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.zip') and 'stresstwin' in f.lower() and 'reproducibility' not in f.lower():
                out = '/kaggle/working/pkg_unzipped'; zipfile.ZipFile(os.path.join(r, f)).extractall(out)
                for r2, d2, f2 in os.walk(out):
                    if 'pyproject.toml' in f2 and os.path.isdir(os.path.join(r2, 'stresstwin_p')):
                        return r2
    print('Nothing found. What is mounted under /kaggle/input:')
    for r, d, f in os.walk('/kaggle/input'):
        if r.count(os.sep) <= 5: print(' ', r, '| files:', f[:6])
    return None
root = find_pkg()
assert root, 'package not found: in the right panel click Add Input -> Your Datasets -> stresstwin-p (it must be listed under Input), then run again'
if not os.path.exists('/kaggle/working/stresstwin_pkg'): shutil.copytree(root, '/kaggle/working/stresstwin_pkg')
subprocess.run([sys.executable,'-m','pip','-q','install','-e','/kaggle/working/stresstwin_pkg'],check=True)
sys.path.insert(0, '/kaggle/working/stresstwin_pkg')   # make the freshly installed package importable in this kernel
WORK='/kaggle/working/work'; os.makedirs(WORK, exist_ok=True)
# /kaggle/working is kept only as notebook OUTPUT. After each stage: Save Version, then make the output a PRIVATE dataset.
# Never make a dataset containing texts.csv public.
def find_input(name, kind='file'):
    """Locate a file or folder by name anywhere under /kaggle/input (the mount path differs between Kaggle versions)."""
    for r, dirs, files in os.walk('/kaggle/input'):
        if kind == 'file' and name in files: return os.path.join(r, name)
        if kind == 'dir' and name in dirs: return os.path.join(r, name)
    raise FileNotFoundError(f'{name} not found under /kaggle/input. Add the dataset that holds it (right panel: Add Input).')


In [ ]:
# StressTwin-F scorer (frozen RoBERTa prefix + trained top-2 block). transformers is pinned to the version the package was built with.
%pip install -q transformers==4.53.2 tokenizers==0.21.4
import numpy as np, pandas as pd
from stresstwin_p.stf_scorer import load_ensemble
from stresstwin_p.score_texts import score_table
EXPORT = find_input('stf_export', 'dir')                    # found wherever Kaggle mounted your stresstwin-f-ckpts dataset
SCORER = 'stresstwin_f'                                     # 'plain_ft' for the ablation run
ens = load_ensemble(f'{EXPORT}/backbone', f'{EXPORT}/ckpt', name=SCORER)
print('seeds', ens.n_seeds, 'device', ens.device)

In [ ]:
# Which texts to score: not the pilot slice (authors whose hash starts with 0 or 1, kept for design decisions) and only authors with
# at least 2 observations (the eligibility rule needs 5; 2 keeps the reply-association threads). This cuts GPU time a lot.
import pandas as pd, numpy as np
obs = pd.read_csv(find_input('observations.csv'))
a = obs.author.astype(str)
keep_ids = set(obs[(~a.str[0].isin(list('01'))) & (obs.groupby('author').obs_id.transform('size') >= 2)].obs_id)
t = pd.read_csv(find_input('texts.csv'))
t = t[t.obs_id.isin(keep_ids)].reset_index(drop=True)
print('texts to score:', len(t), 'of', len(obs))
s = score_table(t.head(100), ens, batch_size=32); print(s.describe().T[['mean','std','min','max']])   # smoke test

In [ ]:
# full run, in chunks so a Colab disconnect loses at most one chunk
import os
CH=20000; os.makedirs(f'{WORK}/scores_{SCORER}', exist_ok=True)
for i in range(0, len(t), CH):
    f=f'{WORK}/scores_{SCORER}/part{i//CH:04d}.csv'
    if os.path.exists(f): continue
    score_table(t.iloc[i:i+CH], ens, batch_size=64).to_csv(f, index=False); print('done', f)

In [ ]:
import glob
pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{WORK}/scores_{SCORER}/part*.csv'))]).to_csv(f'{WORK}/scores_{SCORER}.csv', index=False); print('merged ->', f'{WORK}/scores_{SCORER}.csv')